In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime


In [2]:
# Load raw scraped dataset produced by the scraping pipeline
# This file contains unprocessed product data directly extracted from the website
df = pd.read_csv("../data/raw/whisky_raw.csv")
df.shape


(1023, 27)

In [3]:
# Inspect dataset structure and data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023 entries, 0 to 1022
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   scraped_at                 1023 non-null   object 
 1   category                   1023 non-null   object 
 2   product_id                 1023 non-null   int64  
 3   name                       1023 non-null   object 
 4   brand                      1023 non-null   object 
 5   variant                    1023 non-null   object 
 6   price_ex_vat_gbp           1023 non-null   float64
 7   unit_price_raw             996 non-null    object 
 8   region                     1023 non-null   object 
 9   promo_label                62 non-null     object 
 10  status                     1023 non-null   object 
 11  product_url                1023 non-null   object 
 12  image_url                  1023 non-null   object 
 13  rating_stars               258 non-null    float

In [4]:
# Preview first few rows to verify data loaded correctly
df.head(3)


,scraped_at,category,product_id,name,brand,variant,price_ex_vat_gbp,unit_price_raw,region,promo_label,...,style_body,style_richness,style_smoke,style_sweetness,character_notes,fact_bottler,fact_country,fact_region,fact_cask_type,fact_colouring
0,2026-01-15 09:15:51,Single Malt,3121,Lagavulin 16 Year Old,Lagavulin,70cl / 43%,65.79,(£112.79 per litre),Islay,NaN,...,4.0,3.0,4.0,3.0,Orange|Kippers|Smoke|Brine|Pineapple,Distillery Bottling,Scotland,Islay,NaN,Yes
1,2026-01-15 09:15:51,Single Malt,86756,Thompson Bros Mystery Malt Series No.5,Various Distilleries,70cl / 46.3%,54.12,(£92.79 per litre),Single Malt Scotch Whisky,NaN,...,NaN,NaN,NaN,NaN,NaN,Thompson Bros,Scotland,NaN,NaN,NaN
2,2026-01-15 09:15:51,Single Malt,83347,Aberlour A'Bunadh,Aberlour,70cl / 60%,78.12,(£91.07 per litre),Speyside,Special Offer: £30 Off!,...,4.0,4.0,0.0,3.0,Walnut|Nougat|Orange|Cinnamon,Distillery Bottling,Scotland,Speyside,Oloroso Sherry Butt,NaN


## Data Cleaning 

This section standardizes raw scraped fields into analysis-ready formats. 

In [5]:
# Clean column names
df.columns = df.columns.str.strip()

df.columns


Index(['scraped_at', 'category', 'product_id', 'name', 'brand', 'variant',
       'price_ex_vat_gbp', 'unit_price_raw', 'region', 'promo_label', 'status',
       'product_url', 'image_url', 'rating_stars', 'review_count',
       'price_inc_vat_gbp', 'price_before_discount_gbp', 'style_body',
       'style_richness', 'style_smoke', 'style_sweetness', 'character_notes',
       'fact_bottler', 'fact_country', 'fact_region', 'fact_cask_type',
       'fact_colouring'],
      dtype='object')

In [6]:
# Convert required columns to proper dtypes 

df = df.assign(
    # Datetime
    scraped_at=pd.to_datetime(df["scraped_at"], errors="coerce"),

    # IDs
    product_id=pd.to_numeric(df["product_id"], errors="coerce").astype("Int64"),

    # Prices (numeric)
    price_ex_vat_gbp=pd.to_numeric(df["price_ex_vat_gbp"], errors="coerce"),
    price_inc_vat_gbp=pd.to_numeric(df["price_inc_vat_gbp"], errors="coerce"),
    price_before_discount_gbp=pd.to_numeric(df["price_before_discount_gbp"], errors="coerce"),

    # Ratings (numeric), rating_stars kept as float to preserve half-star granularity (e.g. 4.5)
    rating_stars=pd.to_numeric(df["rating_stars"], errors="coerce"),
    review_count=pd.to_numeric(df["review_count"], errors="coerce").astype("Int64"),

    # Style scores (numeric; keep missing as <NA>)
    style_body=pd.to_numeric(df["style_body"], errors="coerce").astype("Int64"),
    style_richness=pd.to_numeric(df["style_richness"], errors="coerce").astype("Int64"),
    style_smoke=pd.to_numeric(df["style_smoke"], errors="coerce").astype("Int64"),
    style_sweetness=pd.to_numeric(df["style_sweetness"], errors="coerce").astype("Int64"),

    # Strings (keep as string dtype)
    category=df["category"].astype("string"),
    name=df["name"].astype("string"),
    brand=df["brand"].astype("string"),
    variant=df["variant"].astype("string"),
    unit_price_raw=df["unit_price_raw"].astype("string"),
    region=df["region"].astype("string"),
    promo_label=df["promo_label"].astype("string"),
    product_url=df["product_url"].astype("string"),
    image_url=df["image_url"].astype("string"),
    status=df["status"].astype("string"),
    
    character_notes=df["character_notes"].astype("string"),
    fact_bottler=df["fact_bottler"].astype("string"),
    fact_country=df["fact_country"].astype("string"),
    fact_region=df["fact_region"].astype("string"),
    fact_cask_type=df["fact_cask_type"].astype("string"),
    fact_colouring=df["fact_colouring"].astype("string"),
)

# Lightweight validation
print(df.dtypes)
print("\nMissing values after schema enforcement:")
print(df.isna().sum())




scraped_at                   datetime64[ns]
category                     string[python]
product_id                            Int64
name                         string[python]
brand                        string[python]
variant                      string[python]
price_ex_vat_gbp                    float64
unit_price_raw               string[python]
region                       string[python]
promo_label                  string[python]
status                       string[python]
product_url                  string[python]
image_url                    string[python]
rating_stars                        float64
review_count                          Int64
price_inc_vat_gbp                   float64
price_before_discount_gbp           float64
style_body                            Int64
style_richness                        Int64
style_smoke                           Int64
style_sweetness                       Int64
character_notes              string[python]
fact_bottler                 str

In [7]:
# Remove exact duplicate rows and verify product_id uniqueness

before = len(df)

# Define columns to consider for duplicate detection (exclude scraped_at)
dedup_cols = [c for c in df.columns if c != "scraped_at"]

# Drop duplicates based on all columns except scraped_at
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

after = len(df)

# Verify product_id uniqueness
is_unique = df["product_id"].is_unique
dup_count = df["product_id"].duplicated().sum()

print(f"Exact duplicate rows removed: {before - after}")
print(f"product_id unique after dedup: {is_unique}")
print(f"Remaining duplicate product_id count: {dup_count}")

# Enforce one row per product_id (keep the most recent scrape)
before_pid = len(df)

df = (
    df.sort_values("scraped_at")
      .drop_duplicates(subset=["product_id"], keep="last")
      .reset_index(drop=True)
)

after_pid = len(df)

# Verify product_id uniqueness after enforcement
is_unique_final = df["product_id"].is_unique
dup_count_final = df["product_id"].duplicated().sum()

print(f"Rows removed due to duplicate product_id: {before_pid - after_pid}")
print(f"product_id unique after enforcement: {is_unique_final}")
print(f"Remaining duplicate product_id count after enforcement: {dup_count_final}")


Exact duplicate rows removed: 0
product_id unique after dedup: True
Remaining duplicate product_id count: 0
Rows removed due to duplicate product_id: 0
product_id unique after enforcement: True
Remaining duplicate product_id count after enforcement: 0


In [8]:
# Standardize missing values across all text columns

# Exclude URL-like fields from missing token normalization
text_cols = [
    c for c in df.select_dtypes(include="string").columns
    if c not in {"product_url", "image_url"}
]

# Strip leading/trailing whitespace
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

# Standard missing tokens to normalize
MISSING_TOKENS = ["", "none", "n/a", "na", "nan", "null", "undefined"]

for col in text_cols:
    df[col] = df[col].where(
        ~df[col].str.lower().isin(MISSING_TOKENS),
        pd.NA
    )

# Normalize character_notes delimiter formatting (editorial field)
if "character_notes" in df.columns:
    df["character_notes"] = (
        df["character_notes"]
        .str.replace(r"\s*\|\s*", "|", regex=True)
        .str.strip("|")
    )

# Validation
missing_after = df.isna().sum()
print(missing_after[missing_after > 0].sort_values(ascending=False))


price_before_discount_gbp    973
promo_label                  961
review_count                 765
rating_stars                 765
fact_cask_type               756
fact_colouring               533
fact_region                  478
fact_bottler                 471
style_smoke                  470
style_sweetness              466
style_body                   466
style_richness               466
character_notes              441
fact_country                 159
price_inc_vat_gbp            148
unit_price_raw                27
dtype: int64


In [9]:
# Handle inconsistencies with region

# Normalize region text early (strip + lowercase)
df["region"] = (
    df["region"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardize common variants
df["region"] = df["region"].replace({
    "islands": "island",   # TWE sometimes uses plural
    "the islands": "island",
    "n/a": pd.NA
})

# Blended products → region not applicable
df.loc[
    df["category"].str.lower() == "blended",
    "region"
] = pd.NA

# Valid regions for single malt
VALID_REGIONS = [
    "highland", "islay", "speyside", "lowland", "campbeltown", "island"
]

# Single Malt products with non-geographical region labels → set to NA
df.loc[
    (df["category"].str.lower() == "single malt")
    & (~df["region"].isin(VALID_REGIONS)),
    "region"
] = pd.NA

# Final formatting: Capitalize region names for presentation
df["region"] = df["region"].str.capitalize()

# Validation: Ensure only valid regions (and NA) remain
print("Unique regions after cleaning:")
print(df["region"].unique())

# Validation: Check counts to see how many rows were nullified
print("\nRegion value counts (including NAs):")
print(df["region"].value_counts(dropna=False))


Unique regions after cleaning:
<StringArray>
['Islay', 'Highland', 'Speyside', 'Island', <NA>, 'Campbeltown', 'Lowland']
Length: 7, dtype: string

Region value counts (including NAs):
region
<NA>           424
Speyside       230
Highland       153
Islay          106
Island          72
Lowland         26
Campbeltown     12
Name: count, dtype: Int64


In [10]:
# Consolidate region and fact_region into a single region column

# Normalize fact_region text
df["fact_region"] = (
    df["fact_region"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardize common variants in fact_region
df["fact_region"] = df["fact_region"].replace({
    "islands": "island",
    "the islands": "island",
    "n/a": pd.NA
})

# Track how many region values are missing before fill
missing_before = df["region"].isna().sum()

# Fill missing region values using fact_region
df["region"] = df["region"].fillna(df["fact_region"])

# Capitalize final region values for presentation
df["region"] = df["region"].str.capitalize()

# Drop fact_region after consolidation
df = df.drop(columns=["fact_region"])

# Validation
missing_after = df["region"].isna().sum()
filled_count = missing_before - missing_after

print(f"Region values filled from fact_region: {filled_count}")
print("\nFinal region value counts (including NAs):")
print(df["region"].value_counts(dropna=False))


Region values filled from fact_region: 0

Final region value counts (including NAs):
region
<NA>           424
Speyside       230
Highland       153
Islay          106
Island          72
Lowland         26
Campbeltown     12
Name: count, dtype: Int64


In [11]:
# Brand Column Standardization 

# Normalize brand text (strip + collapse repeated whitespace)
df["brand"] = (
    df["brand"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Create a normalized helper key for case-insensitive mapping (logic-only)
brand_key_tmp = df["brand"].str.lower()

# Standardize known aliases / placeholders (case-insensitive)
brand_replacements_ci = {
    "chivas": "Chivas Regal",
    "chivas regal": "Chivas Regal",

    # placeholder / not-a-real-brand style values
    "various distilleries": "Independent/Unknown",
    "undisclosed distillery": "Independent/Unknown",
    "unknown distillery": "Independent/Unknown",
    "secret distillery": "Independent/Unknown",
}

df["brand"] = brand_key_tmp.map(brand_replacements_ci).fillna(df["brand"])

# Flag rows that should be excluded from "price by brand" charts
PLACEHOLDER_BRAND = "Independent/Unknown"
df["is_brand_placeholder"] = df["brand"].eq(PLACEHOLDER_BRAND)

# Validation 
print("\nPlaceholder brand count:")
print(df["is_brand_placeholder"].value_counts(dropna=False))

# Verify Chivas change 
print("\nChivas Regal count:")
print((df["brand"] == "Chivas Regal").sum())



Placeholder brand count:
is_brand_placeholder
False    1018
True        5
Name: count, dtype: int64

Chivas Regal count:
37


In [12]:
# Parse bottle size (cl) and ABV (%) from variant

# Confirm which volume units appear in variant (handles 70cl, 70 cl, 700ml, 1l, etc.)
units = df["variant"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(ml|cl|l|litre|liter)",
    flags=re.IGNORECASE
)[1].str.lower()

print("Volume units found in 'variant':")
print(units.value_counts(dropna=False))

# Extract numeric amount + unit
vol_amount = pd.to_numeric(
    df["variant"].str.extract(r"(\d+(?:\.\d+)?)\s*(?:ml|cl|l|litre|liter)", expand=False),
    errors="coerce"
)

vol_unit = df["variant"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(ml|cl|l|litre|liter)",
    flags=re.IGNORECASE
)[1].str.lower()

# Normalize to Litres (Standard Unit)
df["bottle_size_l"] = pd.NA
df.loc[vol_unit == "ml", "bottle_size_l"] = vol_amount / 1000
df.loc[vol_unit == "cl", "bottle_size_l"] = vol_amount / 100
df.loc[vol_unit.isin(["l", "litre", "liter"]), "bottle_size_l"] = vol_amount

df["bottle_size_l"] = pd.to_numeric(df["bottle_size_l"], errors="coerce")

# Derive cl from litres for convenience
df["bottle_size_cl"] = (df["bottle_size_l"] * 100).round(2)

# Extract ABV (%)
df["abv_percent"] = pd.to_numeric(
    df["variant"].str.extract(r"(\d+(?:\.\d+)?)\s*%", expand=False),
    errors="coerce"
)

# Validation Checks
print(f"Missing Volume (L):  {df['bottle_size_l'].isna().sum()}")
print(f"Missing Volume (cl): {df['bottle_size_cl'].isna().sum()}")
print(f"Missing ABV (%):     {df['abv_percent'].isna().sum()}")

# Preview results
df[["variant", "bottle_size_cl", "bottle_size_l", "abv_percent"]].head()


Volume units found in 'variant':
1
cl    1023
Name: count, dtype: Int64
Missing Volume (L):  0
Missing Volume (cl): 0
Missing ABV (%):     0


,variant,bottle_size_cl,bottle_size_l,abv_percent
0,70cl / 43%,70.0,0.7,43.0
1,70cl / 40%,70.0,0.7,40.0
2,70cl / 40%,70.0,0.7,40.0
3,70cl / 46%,70.0,0.7,46.0
4,70cl / 46%,70.0,0.7,46.0


In [13]:
# VAT fallback: derive missing VAT-inclusive prices from VAT-exclusive prices (UK VAT assumed 20%)

# Calculate only where price_inc_vat_gbp is missing and price_ex_vat_gbp is available
mask_vat_fill = df["price_inc_vat_gbp"].isna() & df["price_ex_vat_gbp"].notna()

df["price_inc_vat_calc_gbp"] = np.nan
df.loc[mask_vat_fill, "price_inc_vat_calc_gbp"] = (df.loc[mask_vat_fill, "price_ex_vat_gbp"] * 1.20).round(2)

# Ensure calc column is numeric (prevents FutureWarning)
df["price_inc_vat_calc_gbp"] = pd.to_numeric(df["price_inc_vat_calc_gbp"], errors="coerce")

# Fill missing VAT-inclusive price using the calculated fallback
df["price_inc_vat_gbp"] = df["price_inc_vat_gbp"].fillna(df["price_inc_vat_calc_gbp"])
df["price_inc_vat_gbp"] = pd.to_numeric(df["price_inc_vat_gbp"], errors="coerce")

# Validation
print("Filled price_inc_vat_gbp using VAT fallback:", int(mask_vat_fill.sum()))
print("Missing price_inc_vat_gbp after fallback:", int(df["price_inc_vat_gbp"].isna().sum()))


Filled price_inc_vat_gbp using VAT fallback: 148
Missing price_inc_vat_gbp after fallback: 0


In [14]:
# Parse unit price from unit_price_raw (standardize to GBP per litre)

# Extract numeric value from unit_price_raw
df["unit_price_gbp_per_litre"] = (
    df["unit_price_raw"]
      .astype("string")
      .str.replace("£", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
)
df["unit_price_gbp_per_litre"] = pd.to_numeric(df["unit_price_gbp_per_litre"], errors="coerce")

# Extract the reported basis after "per" or "/"
basis = (
    df["unit_price_raw"]
      .astype("string")
      .str.lower()
      .str.replace(",", "", regex=False)
      .str.extract(r"(?:per|/)\s*([0-9]+(?:\.[0-9]+)?\s*)?(ml|cl|l|litre|liter)", expand=False)
)

# basis[0] = quantity, basis[1] = unit
qty = pd.to_numeric(basis[0].astype("string").str.strip(), errors="coerce")
unit = basis[1].astype("string").replace({"liter": "litre"})

# If unit is litre/l and qty is missing → treat as 1 litre
mask_litre = ((unit == "litre") | (unit == "l")) & qty.isna()
qty.loc[mask_litre] = 1.0

# Convert all bases to "per litre" using scaling factors
# Default scale = 1.0 so we do NOT destroy extracted numeric values when basis parsing fails
scale = pd.Series(1.0, index=df.index, dtype="float64")

# per X ml  → value * (1000 / X)
mask_ml = (unit == "ml") & qty.notna()
scale.loc[mask_ml] = 1000 / qty.loc[mask_ml]

# per X cl  → value * (100 / X)
mask_cl = (unit == "cl") & qty.notna()
scale.loc[mask_cl] = 100 / qty.loc[mask_cl]

# per litre → value stays the same (or per 2 litres etc → divide by qty)
mask_l = ((unit == "litre") | (unit == "l")) & qty.notna()
scale.loc[mask_l] = 1.0 / qty.loc[mask_l]

# Apply scaling (only changes where basis was understood; otherwise keeps the extracted numeric)
df["unit_price_gbp_per_litre"] = (df["unit_price_gbp_per_litre"] * scale).round(2)

# Preview results
print(f"Rows processed: {len(df)} rows.")
print(f"Missing unit_price_gbp_per_litre: {df['unit_price_gbp_per_litre'].isna().sum()}")

df[["unit_price_raw", "unit_price_gbp_per_litre"]].head(10)

# Fallback: compute unit price per litre when unit_price_raw is missing
mask_unit_fallback = (
    df["unit_price_gbp_per_litre"].isna()
    & df["price_inc_vat_gbp"].notna()
    & df["bottle_size_l"].notna()
    & (df["bottle_size_l"] > 0)
)

df.loc[mask_unit_fallback, "unit_price_gbp_per_litre"] = (
    df.loc[mask_unit_fallback, "price_inc_vat_gbp"]
    / df.loc[mask_unit_fallback, "bottle_size_l"]
).round(2)

print(
    "Filled unit_price_gbp_per_litre using price/volume fallback:",
    int(mask_unit_fallback.sum())
)



Rows processed: 1023 rows.
Missing unit_price_gbp_per_litre: 27
Filled unit_price_gbp_per_litre using price/volume fallback: 27


### Feature Engineering

After completing data cleaning, additional features are created to support analysis. These derived columns help standardize prices, identify discounted products, and group items by characteristics such as bottle size, alcohol strength, and price range. All features are calculated using existing, cleaned data without making assumptions beyond what is available. This ensures that the analysis is clear, accurate, and consistent.

In [15]:
# Price-derived features

# Reference price for analysis:
# - use pre-discount price when available
# - otherwise use current selling price
df["reference_price_gbp"] = df["price_before_discount_gbp"].fillna(df["price_inc_vat_gbp"])

# Total alcohol units per bottle (UK units: units = litres × ABV%)
df["alcohol_units"] = (df["bottle_size_l"] * df["abv_percent"]).round(2)
df.loc[df["alcohol_units"] <= 0, "alcohol_units"] = pd.NA

# Price per alcohol unit (based on reference price)
df["price_per_alcohol_unit_gbp"] = (
    df["reference_price_gbp"] / df["alcohol_units"]
).round(2)

# Outlier flags (added during cleaning; do not remove rows)
# - price outliers: top 1% by price within each category
# - value outliers: top 1% by price_per_alcohol_unit within each category
df["is_price_outlier"] = False
df["is_value_outlier"] = False

for cat in df["category"].dropna().unique():
    mask = df["category"] == cat

    # Price outliers (category-specific)
    prices = df.loc[mask, "price_inc_vat_gbp"].dropna()
    if len(prices) >= 30:
        p99 = prices.quantile(0.99)
        df.loc[mask & (df["price_inc_vat_gbp"] > p99), "is_price_outlier"] = True

    # Value outliers (category-specific)
    values = df.loc[mask, "price_per_alcohol_unit_gbp"].dropna()
    if len(values) >= 30:
        v99 = values.quantile(0.99)
        df.loc[mask & (df["price_per_alcohol_unit_gbp"] > v99), "is_value_outlier"] = True

# Combined outlier flag (convenient single filter in SQL/Power BI)
df["is_outlier"] = df["is_price_outlier"] | df["is_value_outlier"]

# Price tier classification based on reference price
df["price_tier"] = pd.cut(
    df["reference_price_gbp"],
    bins=[0, 40, 120, float("inf")],
    labels=["Budget", "Premium", "Luxury"],
    right=False
)

# Validation
print("Missing reference_price_gbp:", df["reference_price_gbp"].isna().sum())
print("Missing alcohol_units:", df["alcohol_units"].isna().sum())
print("Price tier distribution:")
print(df["price_tier"].value_counts(dropna=False))
print("Outlier counts:")
print("is_price_outlier:", int(df["is_price_outlier"].sum()))
print("is_value_outlier:", int(df["is_value_outlier"].sum()))
print("is_outlier:", int(df["is_outlier"].sum()))


Missing reference_price_gbp: 0
Missing alcohol_units: 0
Price tier distribution:
price_tier
Luxury     469
Premium    462
Budget      92
Name: count, dtype: int64
Outlier counts:
is_price_outlier: 11
is_value_outlier: 11
is_outlier: 12


In [16]:
# Discount-related features (derived from observed prices)

# Ensure price columns are numeric
df["price_inc_vat_gbp"] = pd.to_numeric(df["price_inc_vat_gbp"], errors="coerce")
df["price_before_discount_gbp"] = pd.to_numeric(df["price_before_discount_gbp"], errors="coerce")

# Discount amount (only when both prices exist and the result is positive)
df["discount_amount_gbp"] = df["price_before_discount_gbp"] - df["price_inc_vat_gbp"]
df.loc[df["discount_amount_gbp"] <= 0, "discount_amount_gbp"] = pd.NA

# Discount flag
df["is_discounted"] = df["discount_amount_gbp"].notna()

# Discount percent (0 for non-discounted)
df["discount_percent"] = ((df["discount_amount_gbp"] / df["price_before_discount_gbp"]) * 100).round(2)
df.loc[~df["is_discounted"], "discount_percent"] = 0

# Discount band (Groups discounts into readable ranges)
df["discount_band"] = pd.cut(
    df["discount_percent"],
    bins=[-0.01, 0, 10, 20, 40, float("inf")],
    labels=["No Discount", "Low (≤10%)", "Medium (10–20%)", "High (20–40%)", "Very High (>40%)"]
)

# Quick validation
print("Discounted products:", int(df["is_discounted"].sum()))
print("\nDiscount band distribution:")
print(df["discount_band"].value_counts(dropna=False))

df[[
    "promo_label",
    "price_inc_vat_gbp",
    "price_before_discount_gbp",
    "discount_amount_gbp",
    "discount_percent",
    "discount_band",
    "is_discounted",
]].head(10)


Discounted products: 50

Discount band distribution:
discount_band
No Discount         973
High (20–40%)        21
Medium (10–20%)      19
Low (≤10%)            9
Very High (>40%)      1
Name: count, dtype: int64


,promo_label,price_inc_vat_gbp,price_before_discount_gbp,discount_amount_gbp,discount_percent,discount_band,is_discounted
0,<NA>,78.95,NaN,NaN,0.00,No Discount,False
1,<NA>,47.95,NaN,NaN,0.00,No Discount,False
2,<NA>,40.95,NaN,NaN,0.00,No Discount,False
3,Special Offer: £24 Off!,64.95,88.95,24.0,26.98,High (20–40%),True
4,Special Offer: £4 Off!,42.50,46.50,4.0,8.60,Low (≤10%),True
5,<NA>,56.50,NaN,NaN,0.00,No Discount,False
6,<NA>,52.95,NaN,NaN,0.00,No Discount,False
7,<NA>,59.25,NaN,NaN,0.00,No Discount,False
8,<NA>,110.00,NaN,NaN,0.00,No Discount,False
9,<NA>,87.25,NaN,NaN,0.00,No Discount,False


In [17]:
# Age, volume and ABV related features

# Age statement extraction
df["age_years"] = pd.to_numeric(
    df["name"].astype("string").str.extract(
        r"\b(\d{1,2})\s*(?:year(?:s)?(?:\s+old)?|yo)\b",
        flags=re.IGNORECASE,
        expand=False
    ),
    errors="coerce"
)

# Flag whether age is stated
df["is_age_stated"] = df["age_years"].notna()

# Age bands for age-stated products
df["age_band"] = pd.cut(
    df["age_years"],
    bins=[0, 10, 15, 18, 25, float("inf")],
    labels=["≤10", "11–15", "16–18", "19–25", "25+"]
)

# Bottle size bands (volume-based categorisation)
df["bottle_size_band"] = pd.cut(
    df["bottle_size_l"],
    bins=[0, 0.35, 0.7, 1.0, float("inf")],
    labels=["Mini/Small (≤35cl)", "Half–Standard (35–70cl)", "Standard–Large (70cl–1L)", "Extra Large (>1L)"]
)

# Add bottle size flags (for comparable analysis)
df["is_standard_bottle"] = df["bottle_size_l"] >= 0.70
df["is_comparable_bottle"] = df["bottle_size_l"] >= 0.35

# ABV bands
df["abv_band"] = pd.cut(
    df["abv_percent"],
    bins=[0, 40, 43, 46, 50, float("inf")],
    labels=["≤40%", "40–43%", "43–46%", "46–50%", "50%+"]
)

# Cask strength indicator (derived from ABV threshold)
df["is_cask_strength"] = df["abv_percent"] >= 50

# Validation
print("\nAge stated vs NAS count:")
print(df["is_age_stated"].value_counts(dropna=False))

print("\nAge band distribution (age-stated only):")
print(df.loc[df["is_age_stated"], "age_band"].value_counts())

print("\nBottle size band distribution:")
print(df["bottle_size_band"].value_counts(dropna=False))

print("\nStandard bottle flag count:")
print(df["is_standard_bottle"].value_counts(dropna=False))

print("\nComparable bottle flag count:")
print(df["is_comparable_bottle"].value_counts(dropna=False))

print("\nABV band distribution:")
print(df["abv_band"].value_counts(dropna=False))

print("\nCask strength count:")
print(df["is_cask_strength"].value_counts())



Age stated vs NAS count:
is_age_stated
True     577
False    446
Name: count, dtype: int64

Age band distribution (age-stated only):
age_band
11–15    227
16–18    112
≤10      111
19–25     94
25+       33
Name: count, dtype: int64

Bottle size band distribution:
bottle_size_band
Half–Standard (35–70cl)     750
Standard–Large (70cl–1L)    243
Mini/Small (≤35cl)           17
Extra Large (>1L)            13
Name: count, dtype: int64

Standard bottle flag count:
is_standard_bottle
True     982
False     41
Name: count, dtype: int64

Comparable bottle flag count:
is_comparable_bottle
True     1008
False      15
Name: count, dtype: int64

ABV band distribution:
abv_band
≤40%      296
40–43%    229
50%+      206
43–46%    159
46–50%    133
Name: count, dtype: int64

Cask strength count:
is_cask_strength
False    790
True     233
Name: count, dtype: Int64


In [18]:
# Final column rpuning step


COLUMNS_TO_DROP = [
    "variant",
    "promo_label",
    "unit_price_raw",
    "price_inc_vat_calc_gbp",
]

# Conditionally drop 'status' if it has no variance
if "status" in df.columns and df["status"].nunique(dropna=False) <= 1:
    COLUMNS_TO_DROP.append("status")

# Drop only if columns exist (safe for reruns)
df = df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])

# Final sanity checks
print("Final row count:", df.shape[0])
print("Final column count:", df.shape[1])
print("Dropped columns:", COLUMNS_TO_DROP)

# Confirm no calc/raw helper columns remain
leftover_calc_cols = [c for c in df.columns if "_calc_" in c or c.endswith("_raw")]
print("Remaining _calc_ / _raw_ columns:", leftover_calc_cols)


Final row count: 1023
Final column count: 46
Dropped columns: ['variant', 'promo_label', 'unit_price_raw', 'price_inc_vat_calc_gbp', 'status']
Remaining _calc_ / _raw_ columns: []


#### Saving the Cleaned Dataset

After completing data cleaning and feature engineering, the final processed dataset is saved as a cleaned CSV file. This file serves as the analysis-ready input for all subsequent exploratory analysis and visualizations

In [19]:
# Save cleaned dataframe to processed folder
output_path = "../data/processed/whisky_cleaned.csv"

df.to_csv(output_path, index=False)

print(f"File saved successfully to: {output_path}")




File saved successfully to: ../data/processed/whisky_cleaned.csv
